In [0]:
# ============================================================
# FORGEAXIS
# BRONZE → SILVER PIPELINE
# ============================================================

from pyspark.sql.functions import (
    col,
    trim,
    lower,
    to_date,
    to_timestamp,
    current_timestamp,
    lit
)

from datetime import datetime


# ============================================================
# 1. CONFIGURATION
# ============================================================

CATALOG = "forgeaxis"
BRONZE = "forgeaxis_bronze"
SILVER = "forgeaxis_silver"

# One batch ID for the complete pipeline run
batch_id = datetime.now().strftime("%Y%m%d_%H%M%S")

print("Pipeline Batch ID:", batch_id)


# ============================================================
# 2. CREATE SILVER SCHEMA
# ============================================================

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER}
""")


# ============================================================
# 3. CREATE PIPELINE AUDIT TABLE
# ============================================================

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SILVER}.pipeline_audit (

    batch_id STRING,
    table_name STRING,

    start_time TIMESTAMP,
    end_time TIMESTAMP,

    source_count BIGINT,
    silver_count BIGINT,
    rejected_count BIGINT,

    status STRING,
    error_message STRING

)
USING DELTA
""")


# ============================================================
# 4. CREATE ERROR RECORDS TABLE
# ============================================================

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SILVER}.silver_error_records (

    batch_id STRING,
    table_name STRING,
    record_id STRING,
    error_reason STRING,
    error_time TIMESTAMP

)
USING DELTA
""")


# ============================================================
# 5. FUNCTION TO WRITE SILVER + AUDIT
# ============================================================

def write_silver(
    df,
    source_df,
    table_name,
    start_time
):

    source_count = source_df.count()
    silver_count = df.count()

    rejected_count = source_count - silver_count

    # Add audit columns
    final_df = (
        df
        .withColumn("batch_id", lit(batch_id))
        .withColumn("silver_loaded_at", current_timestamp())
    )

    # Write Silver table
    final_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        f"{CATALOG}.{SILVER}.{table_name}"
    )

    # Write audit information
    spark.sql(f"""
        INSERT INTO {CATALOG}.{SILVER}.pipeline_audit

        VALUES (

            '{batch_id}',
            '{table_name}',

            '{start_time}',
            current_timestamp(),

            {source_count},
            {silver_count},
            {rejected_count},

            'SUCCESS',
            NULL

        )
    """)

    print(
        f"{table_name} : SUCCESS | "
        f"Source={source_count} | "
        f"Silver={silver_count} | "
        f"Rejected={rejected_count}"
    )


# ============================================================
# 6. CUSTOMERS
# ============================================================

try:

    start_time = datetime.now()

    customers_bronze_df = spark.table(
        f"{CATALOG}.{BRONZE}.bronze_customer"
    )

    customers_silver_df = (
        customers_bronze_df

        # Remove duplicate customers
        .dropDuplicates(["customer_id"])

        # Required fields
        .filter(col("customer_id").isNotNull())
        .filter(col("customer_name").isNotNull())

        # Clean strings
        .withColumn(
            "customer_name",
            trim(col("customer_name"))
        )
        .withColumn(
            "city",
            trim(col("city"))
        )
        .withColumn(
            "state",
            trim(col("state"))
        )
        .withColumn(
            "customer_type",
            trim(col("customer_type"))
        )
        .withColumn(
            "status",
            lower(trim(col("status")))
        )

        # Data types
        .withColumn(
            "credit_limit",
            col("credit_limit").cast("double")
        )
        .withColumn(
            "created_at",
            to_timestamp(col("created_at"))
        )

        # Business rule
        .filter(
            col("credit_limit").isNull() |
            (col("credit_limit") >= 0)
        )
    )

    write_silver(
        customers_silver_df,
        customers_bronze_df,
        "silver_customers",
        start_time
    )

except Exception as e:

    error = str(e).replace("'", "''")

    spark.sql(f"""
        INSERT INTO {CATALOG}.{SILVER}.pipeline_audit

        VALUES (
            '{batch_id}',
            'silver_customers',
            '{start_time}',
            current_timestamp(),
            0,
            0,
            0,
            'FAILED',
            '{error}'
        )
    """)

    raise


# ============================================================
# 7. PRODUCTS
# ============================================================

try:

    start_time = datetime.now()

    products_bronze_df = spark.table(
        f"{CATALOG}.{BRONZE}.bronze_products"
    )

    products_silver_df = (
        products_bronze_df

        .dropDuplicates(["product_id"])

        .filter(col("product_id").isNotNull())
        .filter(col("product_name").isNotNull())

        .withColumn(
            "product_name",
            trim(col("product_name"))
        )
        .withColumn(
            "categories",
            trim(col("categories"))
        )
        .withColumn(
            "status",
            lower(trim(col("status")))
        )

        .withColumn(
            "unit_price",
            col("unit_price").cast("double")
        )

        .withColumn(
            "manufacturing_cost",
            col("manufacturing_cost").cast("double")
        )

        .withColumn(
            "created_at",
            to_timestamp(col("created_at"))
        )

        .filter(
            col("unit_price").isNotNull() &
            (col("unit_price") > 0)
        )

        .filter(
            col("manufacturing_cost").isNull() |
            (col("manufacturing_cost") >= 0)
        )
    )

    write_silver(
        products_silver_df,
        products_bronze_df,
        "silver_products",
        start_time
    )

except Exception as e:

    error = str(e).replace("'", "''")

    spark.sql(f"""
        INSERT INTO {CATALOG}.{SILVER}.pipeline_audit

        VALUES (
            '{batch_id}',
            'silver_products',
            '{start_time}',
            current_timestamp(),
            0,
            0,
            0,
            'FAILED',
            '{error}'
        )
    """)

    raise


# ============================================================
# 8. MACHINES
# ============================================================

try:

    start_time = datetime.now()

    machines_bronze_df = spark.table(
        f"{CATALOG}.{BRONZE}.bronze_machine"
    )

    machines_silver_df = (
        machines_bronze_df

        .dropDuplicates(["machine_id"])

        .filter(col("machine_id").isNotNull())

        .withColumn(
            "machine_name",
            trim(col("machine_name"))
        )
        .withColumn(
            "production_line",
            trim(col("production_line"))
        )

        .withColumn(
            "capacity_per_hour",
            col("capacity_per_hour").cast("int")
        )

        .filter(
            col("capacity_per_hour").isNotNull() &
            (col("capacity_per_hour") > 0)
        )
    )

    write_silver(
        machines_silver_df,
        machines_bronze_df,
        "silver_machine",
        start_time
    )

except Exception as e:

    error = str(e).replace("'", "''")

    spark.sql(f"""
        INSERT INTO {CATALOG}.{SILVER}.pipeline_audit

        VALUES (
            '{batch_id}',
            'silver_machine',
            '{start_time}',
            current_timestamp(),
            0,
            0,
            0,
            'FAILED',
            '{error}'
        )
    """)

    raise


# ============================================================
# 9. SALES ORDERS
# ============================================================

try:

    start_time = datetime.now()

    sales_bronze_df = spark.table(
        f"{CATALOG}.{BRONZE}.bronze_sales_orders"
    )

    sales_silver_df = (
        sales_bronze_df

        .dropDuplicates(["sales_order_id"])

        .filter(
            col("sales_order_id").isNotNull()
        )

        .withColumn(
            "quantity_ordered",
            col("quantity_ordered").cast("int")
        )

        .withColumn(
            "unit_price",
            col("unit_price").cast("double")
        )

        .withColumn(
            "total_amount",
            col("total_amount").cast("double")
        )

        .withColumn(
            "order_date",
            to_date(
                col("order_date"),
                "dd-MM-yyyy"
            )
        )

        .withColumn(
            "required_date",
            to_date(
                col("required_date"),
                "dd-MM-yyyy"
            )
        )

        .withColumn(
            "order_status",
            lower(trim(col("order_status")))
        )

        .filter(
            (col("quantity_ordered") > 0) &
            (col("unit_price") > 0) &
            (col("total_amount") >= 0) &
            col("order_date").isNotNull() &
            col("required_date").isNotNull() &
            (
                col("required_date")
                >= col("order_date")
            )
        )
    )

    write_silver(
        sales_silver_df,
        sales_bronze_df,
        "silver_sales_orders",
        start_time
    )

except Exception as e:

    error = str(e).replace("'", "''")

    spark.sql(f"""
        INSERT INTO {CATALOG}.{SILVER}.pipeline_audit

        VALUES (
            '{batch_id}',
            'silver_sales_orders',
            '{start_time}',
            current_timestamp(),
            0,
            0,
            0,
            'FAILED',
            '{error}'
        )
    """)

    raise


# ============================================================
# 10. PRODUCTION ORDERS
# ============================================================

try:

    start_time = datetime.now()

    production_bronze_df = spark.table(
        f"{CATALOG}.{BRONZE}.bronze_production_orders"
    )

    production_silver_df = (
        production_bronze_df

        .dropDuplicates([
            "production_order_id"
        ])

        .filter(
            col("production_order_id").isNotNull()
        )

        .filter(
            col("sales_order_id").isNotNull()
        )

        .filter(
            col("machine_id").isNotNull()
        )

        .withColumn(
            "quantity_produced",
            col("quantity_produced").cast("int")
        )

        .withColumn(
            "start_time",
            to_timestamp(
                col("start_time"),
                "dd-MM-yyyy HH:mm"
            )
        )

        .withColumn(
            "end_time",
            to_timestamp(
                col("end_time"),
                "dd-MM-yyyy HH:mm"
            )
        )

        .withColumn(
            "production_status",
            lower(trim(col("production_status")))
        )

        .filter(
            (col("quantity_produced") >= 0) &
            col("start_time").isNotNull() &
            col("end_time").isNotNull() &
            (
                col("end_time")
                >= col("start_time")
            )
        )
    )

    write_silver(
        production_silver_df,
        production_bronze_df,
        "silver_production_orders",
        start_time
    )

except Exception as e:

    error = str(e).replace("'", "''")

    spark.sql(f"""
        INSERT INTO {CATALOG}.{SILVER}.pipeline_audit

        VALUES (
            '{batch_id}',
            'silver_production_orders',
            '{start_time}',
            current_timestamp(),
            0,
            0,
            0,
            'FAILED',
            '{error}'
        )
    """)

    raise


# ============================================================
# 11. QUALITY CHECKS
# ============================================================

try:

    start_time = datetime.now()

    quality_bronze_df = spark.table(
        f"{CATALOG}.{BRONZE}.bronze_quality_checks"
    )

    quality_silver_df = (
        quality_bronze_df

        .dropDuplicates([
            "quality_check_id"
        ])

        .filter(
            col("quality_check_id").isNotNull()
        )

        .filter(
            col("production_order_id").isNotNull()
        )

        .withColumn(
            "inspector_name",
            trim(col("inspector_name"))
        )

        .withColumn(
            "inspection_time",
            to_timestamp(
                col("inspection_time"),
                "dd-MM-yyyy HH:mm"
            )
        )

        .withColumn(
            "defect_type",
            lower(trim(col("defect_type")))
        )

        .withColumn(
            "defect_count",
            col("defect_count").cast("int")
        )

        .withColumn(
            "quality_status",
            lower(trim(col("quality_status")))
        )

        .filter(
            col("inspection_time").isNotNull() &
            (col("defect_count") >= 0)
        )
    )

    write_silver(
        quality_silver_df,
        quality_bronze_df,
        "silver_quality_checks",
        start_time
    )

except Exception as e:

    error = str(e).replace("'", "''")

    spark.sql(f"""
        INSERT INTO {CATALOG}.{SILVER}.pipeline_audit

        VALUES (
            '{batch_id}',
            'silver_quality_checks',
            '{start_time}',
            current_timestamp(),
            0,
            0,
            0,
            'FAILED',
            '{error}'
        )
    """)

    raise


# ============================================================
# 12. SENSOR LOGS
# ============================================================

try:

    start_time = datetime.now()

    sensor_bronze_df = spark.table(
        f"{CATALOG}.{BRONZE}.bronze_sensor_logs"
    )

    sensor_silver_df = (
        sensor_bronze_df

        .dropDuplicates([
            "sensor_log_id"
        ])

        .filter(
            col("sensor_log_id").isNotNull()
        )

        .filter(
            col("production_order_id").isNotNull()
        )

        .filter(
            col("machine_id").isNotNull()
        )

        .withColumn(
            "timestamp",
            to_timestamp(
                col("timestamp"),
                "dd-MM-yyyy HH:mm"
            )
        )

        .withColumn(
            "temperature",
            col("temperature").cast("double")
        )

        .withColumn(
            "pressure",
            col("pressure").cast("double")
        )

        .withColumn(
            "vibration",
            col("vibration").cast("double")
        )

        .withColumn(
            "humidity",
            col("humidity").cast("double")
        )

        .filter(
            col("timestamp").isNotNull() &
            col("temperature").between(0, 150) &
            (col("pressure") >= 0) &
            (col("vibration") >= 0) &
            col("humidity").between(0, 100)
        )
    )

    write_silver(
        sensor_silver_df,
        sensor_bronze_df,
        "silver_sensor_logs",
        start_time
    )

except Exception as e:

    error = str(e).replace("'", "''")

    spark.sql(f"""
        INSERT INTO {CATALOG}.{SILVER}.pipeline_audit

        VALUES (
            '{batch_id}',
            'silver_sensor_logs',
            '{start_time}',
            current_timestamp(),
            0,
            0,
            0,
            'FAILED',
            '{error}'
        )
    """)

    raise


# ============================================================
# 13. FINAL CHECK
# ============================================================

print("==============================================")
print("FORGEAXIS SILVER PIPELINE COMPLETED")
print("Batch ID:", batch_id)
print("==============================================")

spark.sql(f"""
SELECT *
FROM {CATALOG}.{SILVER}.pipeline_audit
WHERE batch_id = '{batch_id}'
ORDER BY start_time
""").show(truncate=False)

In [0]:
spark.table(
    "forgeaxis.forgeaxis_bronze.bronze_products"
).printSchema()

In [0]:
spark.table(
    "forgeaxis.forgeaxis_silver.silver_products"
).printSchema()

In [0]:
# spark.sql("""
# DROP TABLE IF EXISTS forgeaxis.forgeaxis_silver.silver_products
# """)

In [0]:
spark.table("forgeaxis.forgeaxis_bronze.bronze_machine").printSchema()

spark.table("forgeaxis.forgeaxis_silver.silver_machine").printSchema()